# EXO_03_DARKROOM — Test Rapide TRI-LAYER

```
╔══════════════════════════════════════════════════════════════════════════════╗
║           SCENOGRAPHY DOCK — DARKROOM V1 — TEST RAPIDE TRI-LAYER            ║
║                                                                              ║
║   Mode     : 1 scène (ou N choisies) — validation rapide pipeline           ║
║   Outputs  : environment_*.blend + scene_ready_*.blend dans U04             ║
║   Pipeline : PRODUCTION_PLAN → layer_assembler → .blend → copie U04         ║
║   Stack    : Blender 4.0 Headless + EXO_03_SCENOGRAPHY.py                   ║
╚══════════════════════════════════════════════════════════════════════════════╝
```

**Mode :** Test rapide sur 1 scène. Valide l'injection acteur + copie U04.

## 1. Mount Drive + Configuration

In [ ]:
import os
import sys
import json
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

# === CONFIGURATION ===
DRIVE_ROOT = Path("/content/drive/MyDrive/EXODUS_V2_FRESH")  # <-- Modifier si besoin

# === TEST MODE : limiter les scènes ===
# None = toutes | [1] = première scène seulement | [1, 2] = deux scènes
SCENE_IDS = [1]

# Profil VRAM : colab_t4 (defaut) | colab_a100 | local_low
VRAM_PROFILE = "colab_t4"

# Exposition World Sync
EXPOSURE = 1.0

VERBOSE = True

# ─────────────────────────────────────────────────
UNIT_ROOT       = DRIVE_ROOT / "03_SCENOGRAPHY_DOCK"
CODEBASE        = UNIT_ROOT / "CODEBASE"
IN_CORTEX_JSON  = UNIT_ROOT / "IN_CORTEX_JSON"
IN_MAP_RAW      = UNIT_ROOT / "IN_MAP_RAW"
OUT_PREMIUM     = UNIT_ROOT / "OUT_PREMIUM_SCENE"

sys.path.insert(0, str(CODEBASE))

PRODUCTION_PLAN = IN_CORTEX_JSON / "PRODUCTION_PLAN.JSON"

print("=" * 70)
print("   FRÉGATE 03_SCENOGRAPHY — DARKROOM V1 — TEST RAPIDE")
print("=" * 70)
print(f"\nDrive Root     : {DRIVE_ROOT}")
print(f"CODEBASE       : {CODEBASE}")
print(f"OUT_PREMIUM    : {OUT_PREMIUM}")
print(f"VRAM Profile   : {VRAM_PROFILE}")
print(f"Exposure       : {EXPOSURE}")
print(f"Scènes test    : {SCENE_IDS if SCENE_IDS else 'TOUTES'}")


## 2. Install Blender Headless

In [ ]:
%%bash
BLENDER_BIN=/opt/blender-local/blender
if [ ! -f "$BLENDER_BIN" ]; then
    echo "[DARKROOM] Téléchargement Blender 4.0.2..."
    wget -q https://download.blender.org/release/Blender4.0/blender-4.0.2-linux-x64.tar.xz -O /tmp/blender.tar.xz
    echo "[DARKROOM] Extraction..."
    tar -xf /tmp/blender.tar.xz -C /opt/
    mv /opt/blender-4.0.2-linux-x64 /opt/blender-local
    echo "[DARKROOM] Blender 4.0 installé dans /opt/blender-local"
else
    echo "[DARKROOM] Blender déjà présent : $BLENDER_BIN"
fi
/opt/blender-local/blender --version | head -1
echo "[DARKROOM] GPU disponible :"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "  Pas de GPU NVIDIA"


## 3. Sync CODEBASE depuis GitHub

In [ ]:
# === SYNC CODEBASE depuis GitHub (applique les derniers fixes VULKAN) ===
import subprocess, shutil
from pathlib import Path

REPO_TMP = Path("/tmp/exodus-u03-darkroom-sync")
if REPO_TMP.exists():
    shutil.rmtree(REPO_TMP)

subprocess.run(
    ["git", "clone", "--depth", "1", "https://github.com/kioka8877-ux/EXODUS-V2.git", str(REPO_TMP)],
    check=True, capture_output=True
)

SRC = REPO_TMP / "03_SCENOGRAPHY_DOCK" / "CODEBASE"
DST = CODEBASE
DST.mkdir(parents=True, exist_ok=True)

synced = 0
for f in SRC.iterdir():
    if f.is_file():
        shutil.copy2(str(f), str(DST / f.name))
        synced += 1

print(f"[VULKAN] CODEBASE sync OK — {synced} fichiers depuis GitHub → {DST}")


## 4. Phantom Link

In [ ]:
# Crée les phantom links (lit directement depuis OUT/ des frégates sources)
import subprocess
result = subprocess.run(
    ["python", str(DRIVE_ROOT / "EXO_MARSHAL.py"), "--unit", "U03", "--mode", "link",
     "--drive-root", str(DRIVE_ROOT), "--verbose"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(f"[WARN] Phantom Link retour code {result.returncode} — peut être ignoré si symlinks déjà présents")


## 5. Pre-flight Check

In [ ]:
import shutil as _shutil

print("=" * 60)
print("   PRE-FLIGHT CHECK — DARKROOM U03")
print("=" * 60)

errors = []
warnings = []

# Blender
blender_bin = Path("/opt/blender-local/blender")
if blender_bin.exists():
    print(f"  OK  Blender : {blender_bin}")
else:
    errors.append("Blender introuvable — relancez la cellule d'installation")

# Production Plan
if PRODUCTION_PLAN.exists():
    with open(PRODUCTION_PLAN, "r", encoding="utf-8") as f:
        plan = json.load(f)
    n = len(plan.get("scenes", []))
    print(f"  OK  PRODUCTION_PLAN.JSON : {n} scène(s)")
else:
    errors.append(f"PRODUCTION_PLAN.JSON introuvable : {PRODUCTION_PLAN}")

# ACTOR blend
u04_scene_ref = DRIVE_ROOT / "04_PHOTOGRAPHY_WING" / "IN_SCENE_REF"
actor_blends = list(u04_scene_ref.glob("ACTOR_*.blend")) if u04_scene_ref.exists() else []
if actor_blends:
    print(f"  OK  ACTOR blend auto-détecté : {actor_blends[0].name}")
else:
    warnings.append("Aucun ACTOR_*.blend dans U04/IN_SCENE_REF — acteur absent de la scène")

# HDRi
hdri_found = []
for ext in ("*.hdr", "*.exr", "*.hdri"):
    hdri_found.extend(IN_MAP_RAW.glob(ext))
if hdri_found:
    print(f"  OK  HDRi : {hdri_found[0].name}")
else:
    warnings.append("Aucun HDRi dans IN_MAP_RAW — fallback couleur")

# Disk space
stat = _shutil.disk_usage("/content")
free_gb = stat.free / (1024 ** 3)
print(f"  OK  Espace disque : {free_gb:.1f} GB libre")

print("-" * 60)
if errors:
    for e in errors:
        print(f"  [X] {e}")
    raise RuntimeError(f"{len(errors)} erreur(s) bloquante(s) — corrigez avant de continuer")
else:
    print(f"  OK — Prêt pour assemblage ({len(SCENE_IDS) if SCENE_IDS else 'toutes'} scène(s))")
    for w in warnings:
        print(f"  [!] {w}")


## 6. Lancement Assemblage

In [ ]:
cmd_parts = [
    f"python {CODEBASE}/EXO_03_SCENOGRAPHY.py",
    f"--drive-root {DRIVE_ROOT}",
    f"--production-plan {PRODUCTION_PLAN}",
    f"--blender-path /opt/blender-local/blender",
    f"--vram-profile {VRAM_PROFILE}",
    f"--exposure {EXPOSURE}",
]

if SCENE_IDS:
    scene_str = ",".join(map(str, SCENE_IDS))
    cmd_parts.append(f"--scene-ids {scene_str}")
if VERBOSE:
    cmd_parts.append("-v")

full_cmd = " \\\
    ".join(cmd_parts)
print(f"Commande :\n\n{full_cmd}\n")
print("=" * 70)
print("   DEBUT ASSEMBLAGE TRI-LAYER (DARKROOM)")
print("=" * 70 + "\n")

get_ipython().system(" ".join(cmd_parts))

print("\n" + "=" * 70)
print("   FIN ASSEMBLAGE TRI-LAYER")
print("=" * 70)


## 7. Vérification Output

In [ ]:
print("=" * 60)
print("   VÉRIFICATION OUTPUT — DARKROOM U03")
print("=" * 60)

# .blend produits dans OUT_PREMIUM_SCENE
blends_out = sorted(OUT_PREMIUM.glob("environment_*.blend")) if OUT_PREMIUM.exists() else []
print(f"\n  .blend dans OUT_PREMIUM_SCENE : {len(blends_out)}")
for b in blends_out:
    size_mb = b.stat().st_size / (1024 * 1024)
    print(f"    {b.name} ({size_mb:.1f} MB)")

# assembler_results.json
ar_path = OUT_PREMIUM / "assembler_results.json"
if ar_path.exists():
    with open(ar_path) as f:
        ar = json.load(f)
    print(f"\n  assembler_results.json : {len(ar)} entrée(s)")
    for r in ar:
        sid = r.get("scene_id", "?")
        layers = r.get("layers_active", "")
        actor = r.get("actor_injected", False)
        actor_n = r.get("actor_objects_count", 0)
        actor_status = f"OK ({actor_n} objets)" if actor else "ABSENT"
        print(f"    scene_id={sid} | layers={layers}")
        print(f"      acteur : {actor_status}")
else:
    print("\n  assembler_results.json : ABSENT")

# scene_ready_*.blend copiés vers U04
u04_scene_ref = DRIVE_ROOT / "04_PHOTOGRAPHY_WING" / "IN_SCENE_REF"
scene_ready = sorted(u04_scene_ref.glob("scene_ready_*.blend")) if u04_scene_ref.exists() else []
print(f"\n  scene_ready_*.blend dans U04/IN_SCENE_REF : {len(scene_ready)}")
for b in scene_ready:
    size_mb = b.stat().st_size / (1024 * 1024)
    print(f"    {b.name} ({size_mb:.1f} MB)")

print("\n" + "=" * 60)
if blends_out and scene_ready:
    print("   OK — Assemblage terminé. U04 peut être lancé.")
elif blends_out and not scene_ready:
    print("   WARN — .blend produits mais non copiés vers U04")
    print("   Vérifiez la fonction copy_outputs_to_u04()")
else:
    print("   FAIL — Aucun .blend produit. Vérifiez les logs ci-dessus.")
print("=" * 60)


---
*EXODUS SYSTEM — Fregate 03_SCENOGRAPHY v2.0.0 — DARKROOM Test Mode*